In [4]:
# Q4 – Model complexity and support vectors (RBF SVM)
# Large grid + Examples of underfitting, good fit, overfitting

import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = pd.read_csv("exoplanet.csv")

# --- Log features (same as Q3) ---
for col in ["pl_bmasse", "pl_orbper", "st_rad"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].replace(0, np.nan)

df["log_pl_bmasse"] = np.log(df["pl_bmasse"])
df["log_pl_orbper"] = np.log(df["pl_orbper"])
df["log_st_rad"]    = np.log(df["st_rad"])

feature_cols = [
    "pl_orbeccen",
    "pl_eqt",
    "st_logg",
    "sy_pnum",
    "log_pl_bmasse",
    "log_pl_orbper",
    "log_st_rad",
]

df_model = df[feature_cols + ["planet_class"]].dropna()
df_model[feature_cols] = df_model[feature_cols].fillna(df_model[feature_cols].median())

X = df_model[feature_cols].values
y = df_model["planet_class"].astype(str).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# --- Large grid ---
C_values = np.logspace(-2, 3, 10)     # 0.01 → 1000
gamma_values = np.logspace(-2, 2, 10) # 0.01 → 100

results = []

for C in C_values:
    for gamma in gamma_values:
        svm = SVC(kernel="rbf", C=C, gamma=gamma)
        svm.fit(X_train_s, y_train)

        train_acc = accuracy_score(y_train, svm.predict(X_train_s))
        test_acc  = accuracy_score(y_test, svm.predict(X_test_s))
        sv_count  = svm.n_support_.sum()

        results.append({
            "C": C,
            "gamma": gamma,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "support_vectors": sv_count
        })

results_df = pd.DataFrame(results)

print("=== SUMMARY OF 100 COMBINATIONS ===")
pd.set_option("display.max_rows", None)
print(results_df.to_string())

# --- Identify regimes ---
# Underfitting: both train and test low
underfit = results_df.loc[(results_df["train_acc"] < 0.75) & 
                          (results_df["test_acc"] < 0.75)]

# Good fit: both high and close
goodfit = results_df.loc[(results_df["train_acc"] > 0.85) &
                         (results_df["test_acc"] > 0.85) &
                         (abs(results_df["train_acc"] - results_df["test_acc"]) < 0.03)]

# Overfitting: train very high, test much worse
overfit = results_df.loc[(results_df["train_acc"] > 0.95) &
                         (results_df["test_acc"] < results_df["train_acc"] - 0.07)]


print("\n=== UNDERFITTING EXAMPLES ===")
print(underfit.head())

print("\n=== PROPER FITTING EXAMPLES ===")
print(goodfit.head())

print("\n=== OVERFITTING EXAMPLES ===")
print(overfit.head())

# --- Best configuration ---
best = results_df.loc[results_df["test_acc"].idxmax()]
print("\n=== BEST CONFIGURATION ===")
print(best)


=== SUMMARY OF 100 COMBINATIONS ===
              C       gamma  train_acc  test_acc  support_vectors
0      0.010000    0.010000   0.523641  0.519149             4077
1      0.010000    0.027826   0.616076  0.616312             3783
2      0.010000    0.077426   0.632151  0.633333             3631
3      0.010000    0.215443   0.628369  0.629078             3592
4      0.010000    0.599484   0.616785  0.618440             3796
5      0.010000    1.668101   0.443499  0.452482             3950
6      0.010000    4.641589   0.383452  0.383688             3894
7      0.010000   12.915497   0.383452  0.383688             3905
8      0.010000   35.938137   0.383452  0.383688             4050
9      0.010000  100.000000   0.383452  0.383688             4207
10     0.035938    0.010000   0.626241  0.624113             3722
11     0.035938    0.027826   0.639243  0.642553             3497
12     0.035938    0.077426   0.707801  0.710638             3321
13     0.035938    0.215443   0.741844  